In [224]:
import pandas as pd
from modAL.models import ActiveLearner
from modAL.models import CommitteeRegressor
from modAL.disagreement import vote_entropy_sampling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import torch
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# import kan
from sklearn.metrics import mean_absolute_error, r2_score
from kan import KAN, create_dataset_from_data
# import kan
# from another_kan.pykan.kan import KAN, create_dataset_from_data
# from neizmen_kan.pykan.kan import KAN, create_dataset_from_data

In [96]:
import numpy as np

In [98]:
# torch.autograd.set_detect_anomaly(True)

In [256]:
df = pd.read_csv('ourall.csv')
# for i in df.columns:
#     if df[i].dtype is not np.float64:
#         df[i] = df[i].astype(np.float64)
print(df.columns)
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
# scaler = MinMaxScaler()
# Fit and transform the data
# df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

X_pool = df.drop("PT_LOSS", axis=1).values
y_pool = df["PT_LOSS"].values.reshape(-1, 1)
print(df['B4TOD4'].unique(),'B4TOD4')
print(df['B5TOB4'].unique(),'B5TOB4')
print(df['R45TOB4'].unique(),'R45TOB4')
print(df['OMEGA5'].unique(),'OMEGA5')
print(df['GAMMAU'].unique(),'GAMMAU')
print(df['ALPHA4'].unique(),'ALPHA4')
print(df['RE'].unique(),'RE')

print(len(y_pool))
# print(X_pool,type(X_pool[100]))
# print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_from_data(tX, ty)
# print(tX, ty)

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
[0.01 0.04 0.06 0.08] B4TOD4
[1.25 1.5  1.  ] B5TOB4
[0.9 2.5 5. ] R45TOB4
[-5  0  5] OMEGA5
[-0.5  0.5  0. ] GAMMAU
[10 20 30 40 50 60] ALPHA4
[100000 500000] RE
3888


In [258]:
class KANWrapper:
    def __init__(self, **params):
        self.model = KAN(**params)

    def fit(
        self, X, y, opt="LBFGS", steps=80, lamb=0.05, update_grid=True
    ):  # ,update_grid=False
        dataset = create_dataset_from_data(X, y)
        self.model.fit(
            dataset, opt=opt, steps=steps, lamb=lamb, update_grid=update_grid
        )
        return self

    def predict(self, X):
        probs = self.model(X)
        return probs

In [104]:
def qbc(committee, X_sample):
    loss_fn = torch.nn.MSELoss()
    tx = torch.from_numpy(X_sample).float().requires_grad_(True)

    preds = []
    for model in committee.learner_list:
        pred = model.predict(tx)  # (1, 1) или (1,)
        preds.append(pred)

    preds = torch.stack(preds, dim=0)  # (N_models, 1, D) или (N_models, 1)
    f_avg = preds.mean(dim=0)         # (1, D)
    # MSE между всеми предсказаниями и средним
    loss = loss_fn(preds.squeeze(), f_avg.squeeze())
    return loss, tx

In [106]:
def NA_QBC(committee, X_sample):
    grads = []
    for i in range(X_sample.shape[0]):
        x_s = np.array([X_sample[i]])
        qbc_loss, tx = qbc(committee, x_s)
        if tx.grad is not None:
            tx.grad.zero_()
        qbc_loss.backward(retain_graph=True)
        grads.append(tx.grad.detach().clone())
    return grads

In [108]:
def get_steps(x):
    steps = [{-1:None, 0: 0 , 1: None} for i in range(7)]
# [0.01 0.04 0.06 0.08] B4TOD4
# [1.25 1.5  1.  ] B5TOB4
# [0.9 2.5 5. ] R45TOB4
# [-5  0  5] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10 20 30 40 50 60] ALPHA4
# [100000 500000] RE

    
    if x[0] <= 0.02:
        steps[0][1] = 0.03
        steps[0][-1] = 0
    elif x[0] <= 0.05:
        steps[0][1] = 0.02
        steps[0][-1] = -0.03
    elif x[0] <= 0.07:
        steps[0][1] = 0.02
        steps[0][-1] = -0.02
    elif x[0] > 0.072:
        steps[0][1] = 0
        steps[0][-1] = -0.02

    if x[1] <= 1.15:
        steps[1][1] = 0.25
        steps[1][-1] = 0
    elif x[1] <= 1.4:
        steps[1][1] = 0.25
        steps[1][-1] = -0.25
    elif x[1] > 1.4:
        steps[1][1] = 0
        steps[1][-1] = -0.25

    if x[2] <= 1.15:
        steps[2][1] = 1.6
        steps[2][-1] = 0
    elif x[2] <= 2.7:
        steps[2][1] = 2.5
        steps[2][-1] = -1.6
    elif x[2] > 4.0:
        steps[2][1] = 0
        steps[2][-1] = -2.5

    if x[3] <= -1.15:
        steps[3][1] = 5.0
        steps[3][-1] = 0
    elif x[3] <= 2:
        steps[3][1] = 5.0
        steps[3][-1] = -5.0
    elif x[3] > 3:
        steps[3][1] = 0
        steps[3][-1] = -5.0

    if x[4] <= -0.25:
        steps[4][1] = 0.5
        steps[4][-1] = 0
    elif x[4] <= 0.35:
        steps[4][1] = 0.5
        steps[4][-1] = -0.5
    elif x[4] > 0.4:
        steps[4][1] = 0
        steps[4][-1] = -0.5

    if x[5] <= 15.0:
        steps[5][1] = 10.0
        steps[5][-1] = 0
    elif x[5] <= 25.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 35.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 45.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] <= 55.0:
        steps[5][1] = 10.0
        steps[5][-1] = -10.0
    elif x[5] > 56.0:
        steps[5][1] = 0
        steps[5][-1] = -10.0

    if x[6] <= 150000.:
        steps[6][1] = 400000.
        steps[6][-1] = 0
    elif x[6] >= 400000.:
        steps[6][1] = 0.0
        steps[6][-1] = -400000.
    return steps
        


# [0.005 0.02  0.1  ] B4TOD4
# [1.25 1.   1.5 ] B5TOB4
# [0.9 5.  2.5] R45TOB4
# [-5.  5.  0.] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10. 30. 60. 20. 40. 50.] ALPHA4
# [100000. 500000.] RE

def Lbnd(X_sample, min_max_val =[[0.005,0.1],[1.,1.5],[0.9,5.],[-5.,5.],[-0.5,0.5],[10.,60.],[100000.,500000.]]):
    
    l_grad_bnd = []
    for i in range(X_sample.shape[0]):
        l_grad = []
        x_s = np.array([X_sample[i]])
        for ind in range(len(x_s[0])):
            if  x_s[0][ind] >= min_max_val[ind][1]:
                l_grad.append(-1.0)
            elif min_max_val[ind][0] < x_s[0][ind] < min_max_val[ind][1]:
                l_grad.append(0.0)
            elif  x_s[0][ind] <= min_max_val[ind][0]:
                l_grad.append(1.0)
        l_grad_bnd.append(torch.tensor([l_grad]))
    return l_grad_bnd
    
    

In [366]:
def NA_query_strategy(comittee, X_sample):
    for i in range(4):
        grads = NA_QBC(comittee, X_sample) # (N, M) -> N
        l_grad_bnd = Lbnd(X_sample)
        result = [a - b for a, b in zip(grads, l_grad_bnd)]
        print(result, 'result grads - l_bnd')
        sign = [torch.where(tensor > 0.00001, torch.tensor(1), 
                  torch.where(tensor < 0, torch.tensor(-1), torch.tensor(0))) for tensor in result]
        # sign = [torch.where(tensor > 0, torch.tensor(1)), 
        #   torch.where(tensor < 0, torch.tensor(-1)), torch.where(tensor <= 0.00001, torch.tensor(0)) for tensor in result]
        print(sign, 'sign grads - l_bnd')
        for ind in range(len(sign)):
            real_steps = []
            sign_list = sign[ind].squeeze().tolist()
            steps = get_steps(X_sample[ind])
            x_gen = None
            for index, values in enumerate(sign_list):
                real_steps.append(steps[index][values])
            print(real_steps,'real_steps')
            x_gen = X_sample[ind] + real_steps
            X_sample[ind] = x_gen.copy()
        print(X_sample,"X_sample",i)
    return None, X_sample

In [260]:
def get_new_y(X_sampels,X_pool,y_pool):
    index_list = []
    X_study = []
    y_study = []
    for target_row in X_sampels:
        # print(X_sampels)
        print(target_row)
        # index = np.where((X_pool == target_row).all(axis=1))[0]
        index = np.where(np.all(np.isclose(X_pool, target_row), axis=1))[0]
        # index = np.where(np.all(X_pool == target_row, axis=1))[0]
        s=0
        if index.size > 0:
            # print(f"Найдена строка {target_row} на индексе {index[0]}")
            if index[0] not in index_list:
                X_study.append(target_row)
                y_study.append(y_pool[index[0]])
                index_list.append(index[0])
                X_pool = np.delete(X_pool, index[0], axis=0)
                y_pool = np.delete(y_pool, index[0])
                print(f"Найдена строка {target_row} на индексе {index[0]}")
            else:
                help_copy1 = target_row.copy()
                help_copy2 = target_row.copy()
                while s == 0:
                    i = 0 
                    step1 = get_steps(help_copy1)
                    step2 = get_steps(help_copy2)
                    print(step1,"step")
                    for i in range(7):
                        help_copy1[i] += step1[i][1]
                        help_copy2[i] += step2[i][-1]
                        index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                        if index2.size > 0 and index2[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                            X_study.append(help_copy1)
                            y_study.append(y_pool[index2[0]])
                            index_list.append(index2[0]) 
                            X_pool = np.delete(X_pool, index2[0], axis=0)
                            y_pool = np.delete(y_pool, index2[0])
                            break
                        index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                        if index3.size > 0 and index3[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                            X_study.append(help_copy2)
                            y_study.append(y_pool[index3[0]])
                            index_list.append(index3[0]) 
                            X_pool = np.delete(X_pool, index3[0], axis=0)
                            y_pool = np.delete(y_pool, index3[0])
                            break
                # print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                # X_study.append(help_copy)
                # y_study.append(y_pool[index2[0]])
                # index_list.append(index2[0]) 
                # X_pool = np.delete(X_pool, index2[0], axis=0)
                # y_pool = np.delete(y_pool, index2[0])
                        
                    
        else:
            help_copy1 = target_row.copy()
            help_copy2 = target_row.copy()
            while s == 0:
                i = 0 
                step1 = get_steps(help_copy1)
                step2 = get_steps(help_copy2)
                print(step1,"step1")
                for i in range(7):
                    help_copy1[i] += step1[i][1]
                    help_copy2[i] += step2[i][-1]
                    index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                    if index2.size > 0 and index2[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                        X_study.append(help_copy1)
                        y_study.append(y_pool[index2[0]])
                        index_list.append(index2[0]) 
                        X_pool = np.delete(X_pool, index2[0], axis=0)
                        y_pool = np.delete(y_pool, index2[0])
                        break
                    index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                    if index3.size > 0 and index3[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                        X_study.append(help_copy2)
                        y_study.append(y_pool[index3[0]])
                        index_list.append(index3[0]) 
                        X_pool = np.delete(X_pool, index3[0], axis=0)
                        y_pool = np.delete(y_pool, index3[0])
                        break
            
    return X_study, y_study, index_list

In [368]:
n_members = 4  # количесво моделей
learner_list = list()
grid1 = [3, 7,7,3]
k1 = [5, 3,5,3]
n_queries = 10  # Количество итераций активного обучения
# for i in range(n_queries):
for member_idx in range(n_members):
    # initial training data
    n_initial = 7
    learner = ActiveLearner(
        estimator=KANWrapper(
            width=[7, 7, 7, 1], grid=grid1[member_idx], k=k1[member_idx], seed=42
        ) # вот сюда засовываем наш KAN,X_training=tX, y_training=ty
    )
    learner_list.append(learner)

# assembling the committee

committee = CommitteeRegressor(learner_list=learner_list, query_strategy=NA_query_strategy)

checkpoint directory created: ./model
saving model version 0.0
checkpoint directory created: ./model
saving model version 0.0
checkpoint directory created: ./model
saving model version 0.0
checkpoint directory created: ./model
saving model version 0.0


In [376]:
# n_initial = 7
# np.random.seed(42)
# train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
# X_query = X_pool[train_idx]
# y_query = y_pool[train_idx]
# _, X_generated = committee.query(X_query)

In [250]:
# x_st, y_st, ind = get_new_y(X_generated[0],X_pool,y_pool)
# print(x_st,y_st)
# [0.01 0.04 0.06 0.08] B4TOD4
# [1.25 1.5  1.  ] B5TOB4
# [0.9 2.5 5. ] R45TOB4
# [-5  0  5] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10 20 30 40 50 60] ALPHA4
# [100000 500000] RE


In [370]:
for i in range(120):
    n_initial = 7
    # np.random.seed(42)
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    X_query = X_pool[train_idx]
    y_query = y_pool[train_idx]
    X_pool = np.delete(X_pool, train_idx, axis=0)
    y_pool = np.delete(y_pool, train_idx)
    _, X_generated = committee.query(X_query)
    x_st, y_st, ind = get_new_y(X_generated[0],X_pool,y_pool)
    print(x_st,y_st)
    for i in range(7):
        y_st[i] = np.array([y_st[i]])
    x_st = np.array(x_st)  # Объединяем по оси 0
    y_st = np.array(y_st)
    tX, ty = torch.from_numpy(x_st).float(), torch.from_numpy(y_st).float()
    committee.teach(
        tX,
        ty,
        bootstrap=True,
        update_grid=False
    )


C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:813: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  self.subnode_actscale.append(torch.std(x, dim=0).detach())
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:823: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  input_range = torch.std(preacts, dim=0) + 0.1
C:\Users\ivan\anaconda3\Lib\site-packages\kan\MultKAN.py:824: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\act

[tensor([[ -5.5515, -13.3787, -17.4687,  -7.2845,  21.9446,  51.8743,  19.6839]]), tensor([[-1.1110, -3.2159, -3.9361, -0.9730,  4.3918, 10.3816,  2.7392]]), tensor([[ -5.5518, -16.2125, -19.6690,  -0.8649,  30.6299,  51.8774,  19.6851]]), tensor([[ -5.2892, -16.2116, -16.4254,  -7.2845,  21.9447,  51.8746,  19.6841]]), tensor([[-1.1338, -2.4445, -3.9361, -1.4578,  4.3917, 10.3815,  2.7392]]), tensor([[-1.0894, -2.4446, -4.0871, -0.9730,  1.0290, 10.3817,  2.7392]]), tensor([[ -5.6655, -16.2124, -17.4696,  -0.8649,   9.1388,  51.8770,  19.6849]])] result grads - l_bnd
[tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]]), tensor([[-1, -1, -1, -1,  1,  1,  1]])] sign grads - l_bnd
[-0.02, 0, -2.5, -5.0, 0.5, 10.0, 0.0] real_steps
[-0.02, -0.25, -1.6, 0, 0.5, 10.0, 400000.0] real_steps
[-0.02, -0.25, -1.6

| train_loss: 3.37e-02 | test_loss: 4.00e-02 | reg: 5.63e-01 | : 100%|█| 80/80 [00:17<00:00,  4.57it


saving model version 0.1


| train_loss: 3.18e-02 | test_loss: 3.45e-02 | reg: 7.85e+00 | : 100%|█| 80/80 [00:42<00:00,  1.90it


...

saving model version 0.120


| train_loss: 1.65e-01 | test_loss: 1.56e-01 | reg: 1.72e+01 | : 100%|█| 80/80 [01:04<00:00,  1.24it


saving model version 0.120


| train_loss: 1.56e-01 | test_loss: 1.50e-01 | reg: 2.83e+00 | : 100%|█| 80/80 [00:52<00:00,  1.54it

saving model version 0.120


In [374]:
indx = np.random.choice(range(X_pool.shape[0]), size=100, replace=False)
x_test, y_test = X_pool[indx],y_pool[indx]
tx3,ty3 = torch.from_numpy(x_test).float(), torch.from_numpy(y_test).float()
with torch.set_grad_enabled(False):
    val_preds = committee.predict(tx3)
# print(ty3[:3],val_preds[:3])
y_itog = ty3.numpy()
print(mean_absolute_error(y_itog,val_preds),"mean_absolute_error")
print(r2_score(y_itog,val_preds),'r2_score')

0.09315615300089121 mean_absolute_error
-0.18041965313313568 r2_score


In [184]:
# for i in range(1,7):
#     y_st[i] = np.array([y_st[i]])

# print(x_st,y_st)
# x_st = np.array(x_st)  # Объединяем по оси 0
# y_st = np.array(y_st)
# tX, ty = torch.from_numpy(x_st).float(), torch.from_numpy(y_st).float()
# # print(tX,ty)
# committee.teach(
#     tX,
#     ty,
#     bootstrap=True,
#     update_grid=False
# )

[[ 4.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  3.00e+01  5.00e+05]
 [ 1.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  4.00e+01  5.00e+05]
 [ 1.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  5.00e+01  5.00e+05]
 [ 1.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  6.00e+01  5.00e+05]
 [ 4.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  4.00e+01  5.00e+05]
 [ 4.00e-02  1.00e+00  9.00e-01 -5.00e+00  5.00e-01  5.00e+01  5.00e+05]
 [ 4.00e-02  1.25e+00  9.00e-01 -5.00e+00  5.00e-01  4.00e+01  5.00e+05]] [array([0.0834127]), array([0.0812154]), array([0.0874776]), array([0.0923458]), array([0.0860963]), array([0.0909813]), array([0.0898243])]


| train_loss: 1.86e-02 | test_loss: 1.85e-02 | reg: 9.50e-01 | : 100%|█| 80/80 [00:18<00:00,  4.35it


saving model version 0.1


| train_loss: 1.95e-02 | test_loss: 7.07e-02 | reg: 9.17e+00 | : 100%|█| 80/80 [00:06<00:00, 12.66it

saving model version 0.1


In [79]:
n_members = 2  # количесво моделей
learner_list = list()
grid1 = [3, 7]
k1 = [5, 3]
n_queries = 10  # Количество итераций активного обучения
# for i in range(n_queries):
for member_idx in range(n_members):
    # initial training data
    n_initial = 7
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    X_train = X_pool[train_idx]
    y_train = y_pool[train_idx]
    tX, ty = torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()
    print(X_train,y_train)
    print(tX,ty)
    # creating a reduced copy of the data with the known instances removed
    # X_pool = np.delete(X_pool, train_idx, axis=0)
    # y_pool = np.delete(y_pool, train_idx)

    # initializing learner
    learner = ActiveLearner(
        estimator=KANWrapper(
            width=[7, 7, 7, 1], grid=grid1[member_idx], k=k1[member_idx], seed=42
        ) # вот сюда засовываем наш KAN,X_training=tX, y_training=ty
    )
    learner_list.append(learner)

# assembling the committee

committee = CommitteeRegressor(learner_list=learner_list, query_strategy=NA_query_strategy)

[[ 8.00e-02  1.00e+00  5.00e+00  5.00e+00  5.00e-01  1.00e+01  1.00e+05]
 [ 6.00e-02  1.50e+00  9.00e-01  0.00e+00  5.00e-01  2.00e+01  1.00e+05]
 [ 6.00e-02  1.50e+00  2.50e+00  0.00e+00  5.00e-01  3.00e+01  5.00e+05]
 [ 6.00e-02  1.50e+00  5.00e+00  0.00e+00 -5.00e-01  6.00e+01  5.00e+05]
 [ 1.00e-02  1.25e+00  5.00e+00  5.00e+00  0.00e+00  2.00e+01  5.00e+05]
 [ 8.00e-02  1.00e+00  9.00e-01  5.00e+00 -5.00e-01  3.00e+01  1.00e+05]
 [ 1.00e-02  1.50e+00  9.00e-01 -5.00e+00  0.00e+00  2.00e+01  5.00e+05]] [[0.535913 ]
 [0.128014 ]
 [0.158272 ]
 [0.182592 ]
 [0.204346 ]
 [0.0538867]
 [0.055309 ]]
tensor([[ 8.0000e-02,  1.0000e+00,  5.0000e+00,  5.0000e+00,  5.0000e-01,
          1.0000e+01,  1.0000e+05],
        [ 6.0000e-02,  1.5000e+00,  9.0000e-01,  0.0000e+00,  5.0000e-01,
          2.0000e+01,  1.0000e+05],
        [ 6.0000e-02,  1.5000e+00,  2.5000e+00,  0.0000e+00,  5.0000e-01,
          3.0000e+01,  5.0000e+05],
        [ 6.0000e-02,  1.5000e+00,  5.0000e+00,  0.0000e+00, -5.00

In [83]:
print(y_pool[100],type(y_pool[100]))
print(X_pool[100],type(X_pool[100]))
model1 = KANWrapper(width=[7, 7, 7, 1], grid=3, k=5,seed=42)
model1.fit(tX,ty, update_grid=True, steps=200)


[0.0602579] <class 'numpy.ndarray'>
[1.00e-02 1.25e+00 9.00e-01 5.00e+00 0.00e+00 3.00e+01 1.00e+05] <class 'numpy.ndarray'>
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.73e-01 | test_loss: 1.35e-01 | reg: 1.39e+01 | : 100%|█| 200/200 [00:02<00:00, 75.32

saving model version 0.1


In [266]:
tx3,ty3 = torch.from_numpy(X_pool[100:2000]).float(), torch.from_numpy(y_pool[100:2000]).float()
with torch.set_grad_enabled(False):
    val_preds = committee.predict(tx3)
print(val_preds)
val_preds = learner_list[0].predict(tx3)
print(val_preds[100],ty3)
print(mean_absolute_error(val_preds,ty3),r2_score(val_preds,ty3))

[ 0.10011493 -0.04221142 -0.05963696 ... -0.0051173   0.11976825
 -0.02253521]
tensor([0.0761], grad_fn=<SelectBackward0>) tensor([0.0487, 0.0620, 0.0659,  ..., 0.1010, 0.0794, 0.0898])


RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

In [372]:
indx = np.random.choice(range(X_pool.shape[0]), size=100, replace=False)
x_test, y_test = X_pool[indx],y_pool[indx]
tx3,ty3 = torch.from_numpy(x_test).float(), torch.from_numpy(y_test).float()
with torch.set_grad_enabled(False):
    val_preds = committee.predict(tx3)
# print(ty3[:3],val_preds[:3])
y_itog = ty3.numpy()
print(mean_absolute_error(y_itog,val_preds),"mean_absolute_error")
print(r2_score(y_itog,val_preds),'r2_score')

0.08881851483136416 mean_absolute_error
-0.1615968345469423 r2_score


In [322]:
# ).float(), torch.from_numpy(y_pool[100:2000]).float()
# with torch.set_grad_enabled(False):
#     val_preds = committee.predict(tx3)

In [ ]:
# n_queries = 10  # Количество итераций активного обучения
# n_committee = 5  # Количество моделей в комитете
# # Основной цикл активного обучения
# for i in range(n_queries):
#     # Создание комитета моделей
#     committee = [RandomForestRegressor() for _ in range(n_committee)]
    
#     # Обучение моделей на текущем наборе данных
#     for model in committee:
#         model.fit(X_train, y_train)
    
#     # Получение предсказаний от всех моделей
#     predictions = np.array([model.predict(X_pool) for model in committee])

# # Вычисление неопределенности (разброс предсказаний)
#     uncertainty = np.std(predictions, axis=0)
    
#     # Выбор экземпляра с наибольшей неопределенностью
#     query_index = np.argmax(uncertainty)
    
#     # Добавление выбранного экземпляра в обучающую выборку
#     X_train = np.vstack((X_train, X_pool[query_index].reshape(1, -1)))
#     y_train = np.append(y_train, y_pool[query_index])
    
#     # Удаление выбранного экземпляра из пула
#     X_pool = np.delete(X_pool, query_index, axis=0)
#     y_pool = np.delete(y_pool, query_index)

In [78]:
# def qbc(committee, X_pool, y_pool, n_initial):
#     qsum = [0 for i in range(len(n_initial)]
#     f_average = [0 for i in range(len(n_initial)]
#     train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
#     Sbp = 2
#     X_add = X_pool[train_idx]
#     y_add = y_pool[train_idx]
#     for i in range(n_initial):
#         sample = X_add[i]
#         tx = torch.from_numpy(sample).float() 
#         for model_ind in range(len(committee)):
#             f_average[n_initial] += committee[model_ind].predict(tx)
#         f_average[n_initial] = f_average[n_initial]/len(committee)
#     for i in range(n_initial):
#         sample = X_add[i]
#         for model in committee:
#             qsum[i] += (model.predict(tx) - f_average[i])**2
#         qsum[i] /= len(committee)
#     return qsum